# SWE-Finetune: Multi-Phase Training for SWE-bench & TerminalBench

This notebook trains Qwen3-30B-A3B using Tinker API for optimal performance on:
- **SWE-bench**: Software engineering agent tasks
- **TerminalBench**: Terminal/shell command tasks

## Phases
1. **Coding Foundation** - Magicoder, Evol-Instruct (~155K)
2. **Terminal/Shell** - NL-SHELL-MULTI, NL2SH-ALFA (~145K)
3. **Tool-Use** - xLAM, Glaive function calling (~180K)
4. **SWE-bench Trajectories** - Agent traces (~156K) **CRITICAL**
5. **Competitive Programming** - TACO, CodeForces (~35K)

## 1. Setup

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q tinker tinker-cookbook datasets transformers

In [ ]:
# Clone the repo
!git clone https://github.com/micic-mihajlo/swe-finetune.git /content/swe-finetune 2>/dev/null || echo 'Repo already exists'
import sys
sys.path.insert(0, '/content/swe-finetune')

In [ ]:
# Set Tinker API key
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## 2. Configuration

In [ ]:
#@title Select Training Phase
PHASE = "ALL - Run 1 through 5"  #@param ["1 - Coding Foundation", "2 - Terminal/Shell", "3 - Tool-Use", "4 - SWE-bench (CRITICAL)", "5 - Competitive Programming", "ALL - Run 1 through 5"]
RESUME_FROM_CHECKPOINT = True  #@param {type:"boolean"}
MAX_SAMPLES_PER_DATASET = None  #@param {type:"raw"}

# Parse phase selection
if PHASE.startswith("ALL"):
    PHASES_TO_RUN = [1, 2, 3, 4, 5]
    print("Will run ALL phases sequentially: 1 -> 2 -> 3 -> 4 -> 5")
else:
    PHASES_TO_RUN = [int(PHASE[0])]
    print(f"Will run Phase {PHASES_TO_RUN[0]}: {PHASE}")

In [ ]:
# Load all configs
from configs import (
    phase1_config, phase2_config, phase3_config, 
    phase4_config, phase5_config
)

CONFIGS = {
    1: phase1_config,
    2: phase2_config,
    3: phase3_config,
    4: phase4_config,
    5: phase5_config,
}

print(f"Phases to run: {PHASES_TO_RUN}")
for p in PHASES_TO_RUN:
    c = CONFIGS[p]
    print(f"  Phase {p}: LR={c.training.learning_rate}, Batch={c.training.batch_size}, MaxLen={c.training.max_length}")

## 3. Initialize Tinker

In [ ]:
import tinker
from tinker import types
from tinker.types.tensor_data import TensorData
from transformers import AutoTokenizer
import numpy as np
from tqdm.auto import tqdm

from configs import (
    phase1_config, phase2_config, phase3_config, 
    phase4_config, phase5_config
)
from scripts.data_loaders import (
    load_coding_datasets, load_terminal_datasets, load_tooluse_datasets,
    load_swebench_trajectories, load_competitive_datasets,
)

CONFIGS = {
    1: phase1_config,
    2: phase2_config,
    3: phase3_config,
    4: phase4_config,
    5: phase5_config,
}

LOADERS = {
    1: load_coding_datasets,
    2: load_terminal_datasets,
    3: load_tooluse_datasets,
    4: load_swebench_trajectories,
    5: load_competitive_datasets,
}

# Will hold training client across phases
training_client = None
tokenizer = None

In [ ]:
# Helper: Convert messages to Datum (inline, no imports)
def messages_to_datum(messages, tokenizer, max_length, train_on_last_only=False):
    """Convert conversation messages to Tinker Datum."""
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except:
        parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        text = "\n".join(parts)
    
    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
    if len(tokens) < 2:
        return None
    
    # Create weights
    weights = [0.0] * (len(tokens) - 1)
    
    if train_on_last_only:
        # Only train on last assistant message
        text_decoded = tokenizer.decode(tokens)
        last_assistant_pos = text_decoded.rfind("<|im_start|>assistant")
        if last_assistant_pos != -1:
            prefix_tokens = len(tokenizer.encode(text_decoded[:last_assistant_pos + len("<|im_start|>assistant")]))
            for i in range(prefix_tokens, len(weights)):
                weights[i] = 1.0
    else:
        # Train on all assistant content
        text_decoded = tokenizer.decode(tokens)
        in_assistant = False
        pos = 0
        for i, tok in enumerate(tokens[:-1]):
            tok_text = tokenizer.decode([tok])
            pos += len(tok_text)
            chunk = text_decoded[max(0, pos-50):pos]
            if "<|im_start|>assistant" in chunk:
                in_assistant = True
            elif "<|im_end|>" in tok_text or "<|im_start|>user" in chunk or "<|im_start|>system" in chunk:
                in_assistant = False
            if in_assistant:
                weights[i] = 1.0
    
    # Fallback if no weights set
    if sum(weights) == 0:
        for i in range(len(weights) // 4, len(weights)):
            weights[i] = 1.0
    
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs={
            "target_tokens": TensorData.from_numpy(np.array(target_tokens, dtype=np.int64)),
            "weights": TensorData.from_numpy(np.array(weights, dtype=np.float32)),
        }
    )

def compute_mean_nll(logprobs, weights):
    """Compute mean negative log-likelihood."""
    total_loss = 0.0
    total_weight = 0.0
    for lp, w in zip(logprobs, weights):
        if hasattr(lp, 'to_numpy'):
            lp_arr = lp.to_numpy()
        else:
            lp_arr = np.array(lp)
        if hasattr(w, 'to_numpy'):
            w_arr = w.to_numpy()
        else:
            w_arr = np.array(w)
        total_loss += float(np.sum(-lp_arr * w_arr))
        total_weight += float(np.sum(w_arr))
    return total_loss / total_weight if total_weight > 0 else 0.0

# Create Tinker client
print("Creating Tinker training client...")
service_client = tinker.ServiceClient()

# Main training loop
for phase_num in PHASES_TO_RUN:
    config = CONFIGS[phase_num]
    print(f"\n{'='*60}")
    print(f"PHASE {phase_num}: {config.name}")
    print(f"{'='*60}")
    
    # Create or reuse training client
    if training_client is None:
        training_client = service_client.create_lora_training_client(
            base_model=config.model.name,
            rank=config.model.lora_rank,
        )
        tokenizer = AutoTokenizer.from_pretrained(config.model.name, trust_remote_code=True)
        print(f"Created training client for {config.model.name}")
    
    # Load data
    print(f"Loading data...")
    data_iterator = LOADERS[phase_num](
        streaming=True,
        max_samples_per_dataset=MAX_SAMPLES_PER_DATASET,
        shuffle=True,
    )
    
    # Convert to datums
    print("Converting to datums...")
    datums = []
    train_on_last = (phase_num == 4)  # SWE-bench: only train on last response
    
    for example in tqdm(data_iterator, desc="Processing"):
        messages = example.get("messages", [])
        if messages:
            datum = messages_to_datum(messages, tokenizer, config.training.max_length, train_on_last)
            if datum:
                datums.append(datum)
    
    print(f"Created {len(datums)} datums")
    
    if len(datums) == 0:
        print("No datums created, skipping phase")
        continue
    
    # Training
    n_batches = max(1, len(datums) // config.training.batch_size)
    print(f"Training: {n_batches} batches of {config.training.batch_size}")
    
    np.random.shuffle(datums)
    losses = []
    
    pbar = tqdm(range(n_batches), desc=f"Phase {phase_num} Training")
    for step in pbar:
        batch_start = step * config.training.batch_size
        batch = datums[batch_start:batch_start + config.training.batch_size]
        
        if not batch:
            continue
        
        # Learning rate schedule
        lr_mult = max(0.0, 1.0 - step / n_batches)
        current_lr = config.training.learning_rate * lr_mult
        
        adam_params = types.AdamParams(
            learning_rate=current_lr,
            beta1=0.9,
            beta2=0.95,
            eps=1e-8
        )
        
        # Forward-backward
        fwd_bwd_future = training_client.forward_backward(batch, loss_fn="cross_entropy")
        optim_future = training_client.optim_step(adam_params)
        
        fwd_bwd_result = fwd_bwd_future.result()
        optim_future.result()
        
        # Compute loss
        train_logprobs = [x["logprobs"] for x in fwd_bwd_result.loss_fn_outputs]
        train_weights = [d.loss_fn_inputs["weights"] for d in batch]
        train_nll = compute_mean_nll(train_logprobs, train_weights)
        losses.append(train_nll)
        
        pbar.set_postfix({"NLL": f"{train_nll:.4f}", "LR": f"{current_lr:.2e}"})
        
        # Checkpoint
        if step > 0 and step % config.training.checkpoint_every == 0:
            save_result = training_client.save_state(name=f"{config.name}-{step:06d}").result()
            print(f"\nCheckpoint: {save_result.path}")
    
    # Save phase final
    phase_save = training_client.save_state(name=f"{config.name}-final").result()
    print(f"Phase {phase_num} complete! Final: {phase_save.path}")
    print(f"Avg NLL: {np.mean(losses):.4f}")

print(f"\n{'='*60}")
print("ALL PHASES COMPLETE!")
print(f"{'='*60}")

In [ ]:
# Save final model for inference
final_save = training_client.save_state(name="swe-finetune-final").result()
sampler_save = training_client.save_weights_for_sampler(name="swe-finetune-sampler").result()

print(f"Final state: {final_save.path}")
print(f"Sampler path: {sampler_save.path}")

In [ ]:
# Test inference
sampling_client = service_client.create_sampling_client(model_path=sampler_save.path)

test_prompt = "Write a bash command to find all Python files larger than 1MB"
messages = [{"role": "user", "content": test_prompt}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_tokens = tokenizer.encode(prompt_text)

result = sampling_client.sample(
    prompt=types.ModelInput.from_ints(tokens=prompt_tokens),
    num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=200, temperature=0.7)
).result()

response = tokenizer.decode(result.sequences[0].tokens)
print(f"Prompt: {test_prompt}")
print(f"\nResponse:\n{response}")